# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for a Meal Planner

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [30]:
import os
import json
from dotenv import load_dotenv
import sqlite3
from openai import OpenAI
import gradio as gr

In [6]:

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY.startswith("sk-proj-"):
    print('fine key')
openai = OpenAI()
localai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL_LLAMA = "gemma4:e4b"
test_messages = [{"role":"user","content":"Tell me a snarky joke"}]
response = localai.chat.completions.create(model=MODEL_LLAMA,messages=test_messages)
print(response.choices[0].message.content)

fine key
Here's one for you. Try to keep up, because this might require more critical thinking than you're currently optimizing for.

***

Why did the math book look so sad?

Because it had so many problems.

***

*See?* You're welcome. If that wasn't snarky enough, you probably aren't capable of appreciating the concept of snark in the first place.


In [31]:
system_message = """You are a helpful Healthy Meal Planner Assistant. 
You are always courteous and provide accurate short answers in 1 or 2 lines.
If you do not know the answer you always say so clearly and not make things up. 
Always provide the recipe in a structured format with the ingredients and the steps to prepare the recipe.
Also Incude cost, time to prepare the recipe and calories in the recipe.
You MUST work with the user to plan the meals for the week. If the user does not provide a date, you should ask for a date. 
Save new recipes to the Recipes table and save the meal plan to the meals table using appropriate tools
Make sure to ask for the number of servings for each meal and once the whole week is planned, 
you should display the whole week's plan, total cost of ingredients and total calories.
You MUST ALWAYS use only the provided tools to save the meals and recipes in database.
"""



In [39]:
from sqlite3 import connect

def setup_db():
    conn = connect('meal_planner.db')
    conn.execute('''CREATE TABLE IF NOT EXISTS Recipes 
    (recipe_id INTEGER PRIMARY KEY AUTOINCREMENT,
     recipe_name TEXT NOT NULL,
     ingredients TEXT NOT NULL, -- Can be JSON or plain text
     instructions TEXT NOT NULL,
     prep_time INTEGER,
     cost REAL,
     calories INTEGER)''')
     
     
    conn.execute('''CREATE TABLE IF NOT EXISTS meals
                   (meal_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    meal_type TEXT CHECK(meal_type IN ('Breakfast', 'Lunch', 'Dinner')) NOT NULL,
                    recipe_id INTEGER NOT NULL,
                    date TEXT NOT NULL,
                    notes TEXT,
                    serving_count INTEGER DEFAULT 1,
                    UNIQUE (date, meal_type),
                    FOREIGN KEY (recipe_id) REFERENCES Recipes(recipe_id))
                    ''')

    
    conn.close()

setup_db()

In [40]:
def save_recipe(recipe_name, ingredients, instructions, prep_time, cost, calories):
    conn = connect('meal_planner.db')
    cursor = conn.cursor()
    cursor.execute('''INSERT INTO Recipes (recipe_name, ingredients, instructions, prep_time, cost, calories) VALUES (?, ?, ?, ?, ?, ?) RETURNING recipe_id''', (recipe_name, ingredients, instructions, prep_time, cost, calories))
    result = cursor.fetchone()
    recipe_id = result[0] if result else None
    conn.commit()
    conn.close()
    return recipe_id



def save_meal_plan(meal_type, recipe_id, date, notes, serving_count):
    conn = connect('meal_planner.db')
    conn.execute('''INSERT INTO Meals (meal_type, recipe_id, date, notes, serving_count) VALUES (?, ?, ?, ?, ?)''', (meal_type, recipe_id, date, notes, serving_count))
    conn.commit()
    conn.close()


In [41]:
save_recipe_function = {
    "name": "save_recipe",
    "description": "Add a new recipe to the database",
    "parameters": {
        "type": "object",
        "properties": {
            "recipe_name": {"type": "string", "description": "The name of the recipe"},
            "ingredients": {"type": "string", "description": "The ingredients of the recipe"},
            "instructions": {"type": "string", "description": "The instructions to prepare the recipe"},
            "prep_time": {"type": "integer", "description": "The preparation time of the recipe"},
            "cost": {"type": "float", "description": "The cost of the recipe"},
            "calories": {"type": "integer", "description": "The calories of the recipe"}
        },
        "required": ["recipe_name", "ingredients", "instructions", "prep_time", "cost", "calories"]
    }
}

save_meal_function = {
    "name": "save_meal_plan",
    "description": "Add a new meal to the database",
    "parameters": {
        "type": "object",
        "properties": {
            "meal_type": {"type": "string", "description": "The type of meal (Breakfast, Lunch, Dinner)"},
            "recipe_id": {"type": "integer", "description": "The ID of the recipe to add"},
            "date": {"type": "string", "description": "The date of the meal"},
            "notes": {"type": "string", "description": "Any additional notes about the meal"},
            "serving_count": {"type": "integer", "description": "The number of servings for the meal"}
        },
        "required": ["meal_type", "recipe_id", "date", "notes", "serving_count"]
    }
}

#tools = [set_recipe, set_meal]

tools = [{"type": "function", "function": save_recipe_function},{"type": "function", "function": save_meal_function}]

tools







[{'type': 'function',
  'function': {'name': 'save_recipe',
   'description': 'Add a new recipe to the database',
   'parameters': {'type': 'object',
    'properties': {'recipe_name': {'type': 'string',
      'description': 'The name of the recipe'},
     'ingredients': {'type': 'string',
      'description': 'The ingredients of the recipe'},
     'instructions': {'type': 'string',
      'description': 'The instructions to prepare the recipe'},
     'prep_time': {'type': 'integer',
      'description': 'The preparation time of the recipe'},
     'cost': {'type': 'float', 'description': 'The cost of the recipe'},
     'calories': {'type': 'integer',
      'description': 'The calories of the recipe'}},
    'required': ['recipe_name',
     'ingredients',
     'instructions',
     'prep_time',
     'cost',
     'calories']}}},
 {'type': 'function',
  'function': {'name': 'save_meal_plan',
   'description': 'Add a new meal to the database',
   'parameters': {'type': 'object',
    'propertie

In [ ]:
from datetime import date, timedelta


def getStartDateOfWeek():
    today = date.today()
    # Subtract the number of days already passed in the week
    start_of_week = today - timedelta(days=today.weekday())

    print(f"Current Date: {today}")
    print(f"Start of Week (Monday): {start_of_week}")
    return start_of_week

def chat_local(message,history):
    print(f"Chat Local called with message: {message} and history: {history}")
    
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_message}] + history + [{"role":"user","content":message}]
    response = localai.chat.completions.create(model=MODEL_LLAMA,messages=messages,tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
         
         message = response.choices[0].message
         print(f"Calling Handle tool Calls with {message}")
         responses = handle_tool_calls(message)
         messages.append(message)
         messages.extend(responses)
         print(f'RESPONSES: {responses} \n MESSAGES:{messages}')
         response = localai.chat.completions.create(model=MODEL_LLAMA, messages=messages, tools=tools)
    
    return response.choices[0].message.content
        
       


def handle_tool_calls(message):
    print("Calling tools")
    responses = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        print(f"Tool Name:{tool_name}")
        tool_args = json.loads(tool_call.function.arguments)
        print(f"Tool Args:{tool_args}")
        responses = []
        if tool_name == "save_recipe":
            print(f"Calling save_recipe with {tool_args}")
            recipe_id = save_recipe(**tool_args)
            print(f'RECIPe ID returned:{recipe_id}')
            responses.append({
                "role": "tool",
                "content": f"recipe_id to use for meal plan is {recipe_id}",
                "tool_call_id": tool_call.id
            })
        elif tool_name == "save_meal_plan":
            print(f"Calling add_meal with {tool_args}")
            save_meal_plan(**tool_args)
            date = tool_args.get('date')
            meal_type = tool_args.get('meal_type')

    
            '''responses.append({
                "role": "tool",
                "content": {'date':date, 'meal_type': meal_type},
                "tool_call_id": tool_call.id
            })'''
    return responses

       

def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_message}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create()

In [43]:
view = gr.ChatInterface(fn=chat_local,type="messages")
view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Chat Local called with message: I would like to have strictly vegetarian options for high protein low carb meals and history: []


In [ ]:
DB ="meal_planner.db"

def get_recipe():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM Recipes LIMIT 1')
        result = cursor.fetchone()
        print(f"Recipe Name: ${result[1]}" if result else "No data available for recipe")
    conn.close()

def nuke_recipes():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('DELETE FROM Recipes WHERE 1 = 1')
        conn.commit()
    conn.close()

nuke_recipes()


Chat Local called with message: Monday Apr 6, 2026 and history: [{'role': 'user', 'metadata': None, 'content': 'I would like to have strictly vegetarian options for high protein low carb meals', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'That sounds like a healthy plan! To get started, could you please provide the date you would like the meal plan for, and also let me know the number of servings you usually need for each meal?', 'options': None}]
Chat Local called with message: Suggest an option for breakfast with 2 servings and history: [{'role': 'user', 'metadata': None, 'content': 'I would like to have strictly vegetarian options for high protein low carb meals', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'That sounds like a healthy plan! To get started, could you please provide the date you would like the meal plan for, and also let me know the number of servings you usually need for each meal?', 'options': None}, {'role': 'user'